In [1]:
# =========================
# Cell 0 - Install deps
# =========================
# If you already installed these, you can skip this cell.
%pip install -q google-genai pillow


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
orange3 3.38.1 requires numpy<2,>=1.20.0, but you have numpy 2.2.6 which is incompatible.


In [29]:
from getpass import getpass

API_KEY = getpass("Enter your Gemini API key (input hidden): ").strip()

Enter your Gemini API key (input hidden):  ········


In [47]:
# =========================
# Cell - Parameters (Water texture overlay, NO droplet shapes)
# =========================


MODEL = "gemini-2.5-flash-image"  # or "gemini-3-pro-image-preview"

# --- Water texture overlay (you will apply your own droplet mask later) ---



ASPECT_RATIO = "1:1"          # change if needed: "16:9", "9:16"
IMAGE_SIZE = None             # optional: e.g., "2K" (model-dependent)
RESPONSE_MODALITIES = ["IMAGE"]

INPUT_IMAGE_PATH = None
OUTPUT_DIR = "outputs"
OUTPUT_BASENAME = "wet_glass_texture_overlay"


In [105]:
# --- Water-like shimmer texture for NORMAL alpha blending (full-frame) ---

PROMPT_MAIN = (
    "Create a FULL-FRAME photorealistic WATER CAUSTICS / SHIMMER overlay texture for compositing with normal "
    "alpha blending. The entire image must look like water: rippling refraction lines, caustic patterns, "
    "shimmering highlights, sparkling glints. No horizon, no scene, no objects. This is an overlay texture only. "
    "Keep the base very subtle and transparent-looking (no gray fog layer), with visible highlights and ripples. "
    "No mid-gray base layer."
)

PROMPT_STYLE = (
    "Macro realism, crisp specular highlights, high-frequency ripples, natural randomness, "
    "overlay-friendly, seamless-friendly."
)

NEGATIVE_PROMPT = (
    "horizon, sky, landscape, ocean photo, pool, tiles, objects, people, boats, buildings, "
    "text, watermark, logo, signature, fog, haze, mist, smoke, gray veil, cloudy wash, opaque layer, "
    "low contrast, blurry, cartoon, illustration, repeating pattern, tiled look"
)


In [97]:
# =========================
# Cell 2 - Create client (use GEMINI_API_KEY)
# =========================
import os
from google import genai

os.environ["GEMINI_API_KEY"] = API_KEY
client = genai.Client()


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [98]:
# =========================
# Cell 3 - Build config (IMPORTANT: response_modalities)
# =========================
from google.genai import types

def build_config(aspect_ratio: str | None, image_size: str | None, response_modalities: list[str]):
    img_cfg = types.ImageConfig(aspect_ratio=aspect_ratio)
    if image_size:
        img_cfg = types.ImageConfig(aspect_ratio=aspect_ratio, image_size=image_size)

    return types.GenerateContentConfig(
        response_modalities=response_modalities,
        image_config=img_cfg,
    )


In [101]:
# =========================
# Cell 4 - Generate
# =========================
from pathlib import Path
from datetime import datetime
from PIL import Image
from IPython.display import display

def compose_prompt(main: str, style: str | None, negative: str | None) -> str:
    parts = [main.strip()]
    if style and style.strip():
        parts.append(f"Style constraints: {style.strip()}")
    if negative and negative.strip():
        parts.append(f"Avoid: {negative.strip()}")
    # make it explicit we want image output
    parts.append("Return the image output.")
    return "\n\n".join(parts)

def normalize_model_name(name: str) -> str:
    name = (name or "").strip()
    return name[len("models/"):] if name.startswith("models/") else name

model_name = normalize_model_name(MODEL)
final_prompt = compose_prompt(PROMPT_MAIN, PROMPT_STYLE, NEGATIVE_PROMPT)

config = build_config(ASPECT_RATIO, IMAGE_SIZE, RESPONSE_MODALITIES)

contents = [final_prompt]

if INPUT_IMAGE_PATH:
    img = Image.open(INPUT_IMAGE_PATH)
    contents = [final_prompt, img]

response = client.models.generate_content(
    model=model_name,
    contents=contents,
    config=config,
)

# Parse response.parts (per official examples)
images = []
texts = []
for part in response.parts:
    if part.text is not None:
        texts.append(part.text)
    elif part.inline_data is not None:
        images.append(part.as_image())

print("Model:", model_name)
print("Text parts returned:", len(texts))
print("Images returned:", len(images))

if not images:
    # Try fallback: allow both TEXT+IMAGE, some responses may include both
    raise RuntimeError(
        "No images returned. Try:\n"
        "1) RESPONSE_MODALITIES = ['TEXT','IMAGE']\n"
        "2) MODEL = 'gemini-3-pro-image-preview'\n"
        "3) Ensure your prompt starts with 'Generate an image...'\n"
    )

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
saved_paths = []

for idx, im in enumerate(images, start=1):
    out_path = out_dir / f"{OUTPUT_BASENAME}_{ts}_{idx:02d}.png"
    im.save(out_path)
    saved_paths.append(out_path)
    display(im)

print("\nSaved files:")
for p in saved_paths:
    print(" -", p.as_posix())


Model: gemini-2.5-flash-image
Text parts returned: 0
Images returned: 1


Image(
  image_bytes=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x04\x00\x00\x00\x04\x00\x08\x02\x00\x00\x00\xf0\x7f\xbc\xd4\x00\x00\x1a\xafcaBX\x00\x00\x1a\xafjumb\x00\x00\x00\x1ejumdc2pa\x00\x11\x00\x10\x80\x00\x00\xaa\x008\x9bq\x03c2pa\x00\x00\x00\x1a\x89jumb\x00\x00\x00Gjumdc2...',
  mime_type='image/png'
)


Saved files:
 - outputs/wet_glass_texture_overlay_20260101_113722_01.png


In [ ]:
# =========================
# Cell - Generate 10 VARIATIONS (same base prompts, each output slightly different)
# =========================
import random
from pathlib import Path
from datetime import datetime
from IPython.display import display

N_IMAGES = 10

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

saved_paths = []

for i in range(N_IMAGES):
    # Same base prompt content, but we append an invisible-to-you "variation tag"
    # that nudges the model to sample differently WITHOUT changing the described intent.
    variation_tag = f"Variation-ID: {random.randint(100000, 999999)}"

    contents = (
        f"{PROMPT_MAIN}\n\n"
        f"Style constraints: {PROMPT_STYLE}\n\n"
        f"Avoid: {NEGATIVE_PROMPT}\n\n"
        f"Make this output different from previous ones while keeping the same style. {variation_tag}\n"
        f"Return the image output."
    )

    response = client.models.generate_content(
        model=normalize_model_name(MODEL),
        contents=[contents],
        config=build_config(ASPECT_RATIO, IMAGE_SIZE, RESPONSE_MODALITIES),
    )

    img = None
    for part in response.parts:
        if part.inline_data is not None:
            img = part.as_image()
            break

    if img is None:
        print(f"[{i+1}/{N_IMAGES}] No image returned (skipping).")
        continue

    out_path = out_dir / f"caustics_overlay_{ts}_{i+1:02d}.png"
    img.save(out_path)
    saved_paths.append(out_path)

    print(f"[{i+1}/{N_IMAGES}] Saved: {out_path.as_posix()}")
    display(img)

print("\nDone. Total saved:", len(saved_paths))


[1/10] Saved: outputs/caustics_overlay_20260101_113938_01.png


Image(
  image_bytes=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x04\x00\x00\x00\x04\x00\x08\x02\x00\x00\x00\xf0\x7f\xbc\xd4\x00\x00\x1b\xd6caBX\x00\x00\x1b\xd6jumb\x00\x00\x00\x1ejumdc2pa\x00\x11\x00\x10\x80\x00\x00\xaa\x008\x9bq\x03c2pa\x00\x00\x00\x1b\xb0jumb\x00\x00\x00Gjumdc2...',
  mime_type='image/png'
)

[2/10] Saved: outputs/caustics_overlay_20260101_113938_02.png


Image(
  image_bytes=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x04\x00\x00\x00\x04\x00\x08\x02\x00\x00\x00\xf0\x7f\xbc\xd4\x00\x00\x1b\x8bcaBX\x00\x00\x1b\x8bjumb\x00\x00\x00\x1ejumdc2pa\x00\x11\x00\x10\x80\x00\x00\xaa\x008\x9bq\x03c2pa\x00\x00\x00\x1bejumb\x00\x00\x00Gjumdc2...',
  mime_type='image/png'
)

[3/10] Saved: outputs/caustics_overlay_20260101_113938_03.png


...

Image(
  image_bytes=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x04\x00\x00\x00\x04\x00\x08\x02\x00\x00\x00\xf0\x7f\xbc\xd4\x00\x00\x1a}caBX\x00\x00\x1a}jumb\x00\x00\x00\x1ejumdc2pa\x00\x11\x00\x10\x80\x00\x00\xaa\x008\x9bq\x03c2pa\x00\x00\x00\x1aWjumb\x00\x00\x00Gjumdc2...',
  mime_type='image/png'
)

[8/10] Saved: outputs/caustics_overlay_20260101_113938_08.png


Image(
  image_bytes=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x04\x00\x00\x00\x04\x00\x08\x02\x00\x00\x00\xf0\x7f\xbc\xd4\x00\x00\x1b\x1dcaBX\x00\x00\x1b\x1djumb\x00\x00\x00\x1ejumdc2pa\x00\x11\x00\x10\x80\x00\x00\xaa\x008\x9bq\x03c2pa\x00\x00\x00\x1a\xf7jumb\x00\x00\x00Gjumdc2...',
  mime_type='image/png'
)

[9/10] Saved: outputs/caustics_overlay_20260101_113938_09.png


Image(
  image_bytes=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x04\x00\x00\x00\x04\x00\x08\x02\x00\x00\x00\xf0\x7f\xbc\xd4\x00\x00\x1bmcaBX\x00\x00\x1bmjumb\x00\x00\x00\x1ejumdc2pa\x00\x11\x00\x10\x80\x00\x00\xaa\x008\x9bq\x03c2pa\x00\x00\x00\x1bGjumb\x00\x00\x00Gjumdc2...',
  mime_type='image/png'
)